# 2장. 백엔드·서버 연습 (HTTP · REST · 인증 · 캐시 · Django ORM · Linux · Docker · AWS)

서버나 Docker를 띄우지 않고 노트북 안에서 손으로 확인할 수 있는 것만 골랐어요.
- HTTP·상태코드 → **FastAPI + TestClient**로 진짜 요청/응답을 만들어 봐요
- Django ORM N+1 → SQLite에 나간 **쿼리 개수를 세서** 확인해요
- Linux → `subprocess`로 실제 셸 명령을 돌려요 (macOS/Linux)
- Docker·AWS → Dockerfile 검사기와 퀴즈로 연습해요

In [ ]:
# ▶ 먼저 이 셀을 실행하세요 (채점 도우미)
import os
STRICT = os.environ.get("PRACTICE_STRICT") == "1"
___ = None  # 빈칸 표시. 이 자리를 채우면 돼요

def check(name, cond, hint=""):
    """cond가 참이면 ✅, 아니면 ❌ (cond는 값이나 lambda)"""
    try:
        ok = bool(cond() if callable(cond) else cond)
    except Exception as e:
        ok, hint = False, f"{type(e).__name__}: {e}"
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"  → {hint}"))
    if STRICT and not ok:
        raise AssertionError(name)

## 2-1. 상태 코드 · REST · 멱등성

In [ ]:
status = {
    "주문 조회 성공":                          ___,
    "새 환불 생성 성공":                        ___,
    "JSON 필드 형식이 틀림 (FastAPI 검증 실패)": ___,
    "로그인 안 함 (누구인지 모름)":              ___,
    "로그인은 했는데 남의 주문을 보려 함":        ___,
    "없는 주문 번호":                           ___,
    "이미 환불된 주문을 또 환불":                ___,
    "1분에 요청을 너무 많이 보냄":               ___,
    "서버 코드에서 예외 발생":                   ___,
}

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
status = {
    "주문 조회 성공":                          200,
    "새 환불 생성 성공":                        201,
    "JSON 필드 형식이 틀림 (FastAPI 검증 실패)": 422,
    "로그인 안 함 (누구인지 모름)":              401,
    "로그인은 했는데 남의 주문을 보려 함":        403,
    "없는 주문 번호":                           404,
    "이미 환불된 주문을 또 환불":                409,
    "1분에 요청을 너무 많이 보냄":               429,
    "서버 코드에서 예외 발생":                   500,
}
```
</details>

In [ ]:
check("상태 코드", lambda: list(status.values()) == [200, 201, 422, 401, 403, 404, 409, 429, 500])

REST: 자원은 **URL 명사**, 동작은 **메서드**로. `(메서드, 경로)` 튜플로 적어요.

In [ ]:
rest = {
    "1003번 주문 조회":              (___, ___),
    "1003번 주문에 환불 생성":        (___, ___),
    "1003번 주문의 배송지만 수정":     (___, ___),
    "1003번 주문 삭제":              (___, ___),
}
# 여러 번 보내도 결과가 같은(멱등) 메서드를 모두
idempotent = {___}

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
rest = {
    "1003번 주문 조회":              ("GET", "/orders/1003"),
    "1003번 주문에 환불 생성":        ("POST", "/orders/1003/refunds"),
    "1003번 주문의 배송지만 수정":     ("PATCH", "/orders/1003"),
    "1003번 주문 삭제":              ("DELETE", "/orders/1003"),
}
idempotent = {"GET", "PUT", "DELETE"}
```
</details>

In [ ]:
check("REST 설계", lambda: list(rest.values()) == [("GET", "/orders/1003"), ("POST", "/orders/1003/refunds"),
                                                ("PATCH", "/orders/1003"), ("DELETE", "/orders/1003")])
check("멱등 메서드", lambda: idempotent == {"GET", "PUT", "DELETE"})

## 2-2. 미니 API 만들기 (FastAPI)
아래 API의 `___`에 **상태 코드**를 채워요. 요청 헤더 `X-User`가 로그인한 회원 id라고 칠게요.

| 상황 | 응답 |
|---|---|
| `X-User` 헤더 없음 | 401 |
| 주문 없음 | 404 |
| 남의 주문 | 403 |
| 환불 생성 성공 | 201 |
| 이미 환불됨 | 409 |

In [ ]:
from fastapi import FastAPI, HTTPException, Header
from fastapi.testclient import TestClient

ORDERS = {1003: {"id": 1003, "user": "7", "status": "paid"},
          1004: {"id": 1004, "user": "3", "status": "paid"}}
app = FastAPI()

def load_order(order_id, x_user):
    if x_user is None:
        raise HTTPException(status_code=___, detail="로그인이 필요해요")
    order = ORDERS.get(order_id)
    if order is None:
        raise HTTPException(status_code=___, detail="주문이 없어요")
    if order["user"] != x_user:
        raise HTTPException(status_code=___, detail="권한이 없어요")
    return order

@app.get("/orders/{order_id}")
def get_order(order_id: int, x_user: str | None = Header(default=None)):
    return load_order(order_id, x_user)

@app.post("/orders/{order_id}/refunds", status_code=___)
def create_refund(order_id: int, x_user: str | None = Header(default=None)):
    order = load_order(order_id, x_user)
    if order["status"] == "refunded":
        raise HTTPException(status_code=___, detail="이미 환불됐어요")
    order["status"] = "refunded"
    return {"order_id": order_id, "status": "refunded"}

client = TestClient(app)

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
from fastapi import FastAPI, HTTPException, Header
from fastapi.testclient import TestClient

ORDERS = {1003: {"id": 1003, "user": "7", "status": "paid"},
          1004: {"id": 1004, "user": "3", "status": "paid"}}
app = FastAPI()

def load_order(order_id, x_user):
    if x_user is None:
        raise HTTPException(status_code=401, detail="로그인이 필요해요")
    order = ORDERS.get(order_id)
    if order is None:
        raise HTTPException(status_code=404, detail="주문이 없어요")
    if order["user"] != x_user:
        raise HTTPException(status_code=403, detail="권한이 없어요")
    return order

@app.get("/orders/{order_id}")
def get_order(order_id: int, x_user: str | None = Header(default=None)):
    return load_order(order_id, x_user)

@app.post("/orders/{order_id}/refunds", status_code=201)
def create_refund(order_id: int, x_user: str | None = Header(default=None)):
    order = load_order(order_id, x_user)
    if order["status"] == "refunded":
        raise HTTPException(status_code=409, detail="이미 환불됐어요")
    order["status"] = "refunded"
    return {"order_id": order_id, "status": "refunded"}

client = TestClient(app)
```
</details>

In [ ]:
me = {"X-User": "7"}
check("헤더 없음 → 401", lambda: client.get("/orders/1003").status_code == 401)
check("내 주문 → 200",   lambda: client.get("/orders/1003", headers=me).status_code == 200)
check("남의 주문 → 403", lambda: client.get("/orders/1004", headers=me).status_code == 403)
check("없는 주문 → 404", lambda: client.get("/orders/9999", headers=me).status_code == 404)
check("숫자 아닌 id → 422 (자동 검증)", lambda: client.get("/orders/abc", headers=me).status_code == 422)
check("환불 생성 → 201", lambda: client.post("/orders/1003/refunds", headers=me).status_code == 201)
check("또 환불 → 409",   lambda: client.post("/orders/1003/refunds", headers=me).status_code == 409)
print(client.get("/orders/1004", headers=me).json())

## 2-3. 세션 vs JWT
JWT의 핵심은 **서명**이에요. 서버만 아는 비밀키로 `내용`에 서명해 두면, 누가 내용을 고쳐도 서명이 안 맞아서 들통나요.
아래 `sign()`을 보고 `verify()`를 완성해요 (진짜 JWT를 단순화한 버전).
- 서명이 안 맞으면 `None`
- `exp`(만료 시각)가 `now`보다 이전이면 `None`
- 통과하면 payload dict 반환

In [ ]:
import hmac, hashlib, base64, json

SECRET = b"dev-only-secret"
b64   = lambda b: base64.urlsafe_b64encode(b).rstrip(b"=").decode()
unb64 = lambda s: base64.urlsafe_b64decode(s + "=" * (-len(s) % 4))

def sign(payload):
    body = b64(json.dumps(payload).encode())
    sig = b64(hmac.new(SECRET, body.encode(), hashlib.sha256).digest())
    return f"{body}.{sig}"

def verify(token, now):
    body, sig = token.split(".")
    expected = ___                                  # sign()과 같은 방법으로 서명 다시 계산
    if not hmac.compare_digest(sig, expected):
        return None
    payload = ___                                   # body → dict
    if ___:                                         # 만료 검사
        return None
    return payload

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
import hmac, hashlib, base64, json

SECRET = b"dev-only-secret"
b64   = lambda b: base64.urlsafe_b64encode(b).rstrip(b"=").decode()
unb64 = lambda s: base64.urlsafe_b64decode(s + "=" * (-len(s) % 4))

def sign(payload):
    body = b64(json.dumps(payload).encode())
    sig = b64(hmac.new(SECRET, body.encode(), hashlib.sha256).digest())
    return f"{body}.{sig}"

def verify(token, now):
    body, sig = token.split(".")
    expected = b64(hmac.new(SECRET, body.encode(), hashlib.sha256).digest())
    if not hmac.compare_digest(sig, expected):
        return None
    payload = json.loads(unb64(body))
    if payload["exp"] < now:
        return None
    return payload
```
</details>

In [ ]:
tok = sign({"user": 7, "role": "user", "exp": 1000})
body, sig = tok.split(".")
fake_body = b64(json.dumps({"user": 7, "role": "admin", "exp": 1000}).encode())   # 권한을 몰래 admin으로
check("정상 토큰", lambda: verify(tok, now=500) == {"user": 7, "role": "user", "exp": 1000})
check("내용 조작 → 거부", lambda: verify(f"{fake_body}.{sig}", now=500) is None)
check("만료 → 거부", lambda: verify(tok, now=2000) is None)

In [ ]:
# 세션과 JWT 비교: "세션" 또는 "JWT"
auth_quiz = {
    "서버가 로그인 상태를 저장하고, 브라우저는 ID 쿠키만 가진다": ___,
    "서버 저장소 없이 여러 서버가 같은 비밀키로 검증할 수 있다": ___,
    "로그아웃·강제 만료를 바로 하기 쉽다": ___,
}
# "인증" 또는 "인가"
authn_authz = {"누구인지 확인 (로그인)": ___, "이 사람이 이걸 해도 되는지 (권한)": ___}

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
auth_quiz = {
    "서버가 로그인 상태를 저장하고, 브라우저는 ID 쿠키만 가진다": "세션",
    "서버 저장소 없이 여러 서버가 같은 비밀키로 검증할 수 있다": "JWT",
    "로그아웃·강제 만료를 바로 하기 쉽다": "세션",   # 서버에서 세션을 지우면 끝
}
authn_authz = {"누구인지 확인 (로그인)": "인증", "이 사람이 이걸 해도 되는지 (권한)": "인가"}
```
</details>

In [ ]:
check("세션 vs JWT", lambda: list(auth_quiz.values()) == ["세션", "JWT", "세션"])
check("인증 vs 인가 (401 vs 403과 짝)", lambda: list(authn_authz.values()) == ["인증", "인가"])

## 2-4. 캐시 (만료 시간이 핵심)
Redis의 `SET key value EX 60` 같은 **TTL 캐시**를 직접 만들어요.
테스트하기 쉽게 시계를 `clock` 함수로 받아요.

In [ ]:
class TTLCache:
    def __init__(self, ttl, clock):
        self.ttl, self.clock, self.data = ttl, clock, {}

    def set(self, key, value):
        self.data[key] = (value, ___)            # 값과 "만료 시각"을 같이 저장

    def get(self, key):
        if key not in self.data:
            return None
        value, expires_at = self.data[key]
        if ___:                                  # 만료됐으면 지우고 None
            del self.data[key]
            return None
        return value

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
class TTLCache:
    def __init__(self, ttl, clock):
        self.ttl, self.clock, self.data = ttl, clock, {}

    def set(self, key, value):
        self.data[key] = (value, self.clock() + self.ttl)

    def get(self, key):
        if key not in self.data:
            return None
        value, expires_at = self.data[key]
        if self.clock() >= expires_at:
            del self.data[key]
            return None
        return value
```
</details>

In [ ]:
now = [0]
cache = TTLCache(ttl=60, clock=lambda: now[0])
cache.set("faq:배송", "결제 후 2~3일")
now[0] = 30
check("만료 전엔 값이 있다", lambda: cache.get("faq:배송") == "결제 후 2~3일")
now[0] = 61
check("만료 후엔 None", lambda: cache.get("faq:배송") is None)
check("만료되면 지워진다", lambda: "faq:배송" not in cache.data)

## 2-5. 동기 vs 비동기
LLM API 호출처럼 **기다리는 시간이 긴** 일은 `async/await`로 동시에 기다릴 수 있어요.
0.3초 걸리는 가짜 LLM 호출 5개를 순서대로 vs 동시에(`asyncio.gather`) 실행해 시간을 비교해요.

In [ ]:
import asyncio, time

async def fake_llm(question):
    await asyncio.sleep(0.3)        # 네트워크 대기라고 생각
    return f"답변({question})"

questions = ["배송", "환불", "교환", "결제", "회원"]

t = time.perf_counter()
seq = [await fake_llm(x) for x in questions]
t_seq = time.perf_counter() - t

t = time.perf_counter()
par = await ___                     # 5개를 동시에
t_par = time.perf_counter() - t
print(f"순서대로 {t_seq:.2f}초 / 동시에 {t_par:.2f}초")

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
import asyncio, time

async def fake_llm(question):
    await asyncio.sleep(0.3)        # 네트워크 대기라고 생각
    return f"답변({question})"

questions = ["배송", "환불", "교환", "결제", "회원"]

t = time.perf_counter()
seq = [await fake_llm(x) for x in questions]
t_seq = time.perf_counter() - t

t = time.perf_counter()
par = await asyncio.gather(*(fake_llm(x) for x in questions))
t_par = time.perf_counter() - t
print(f"순서대로 {t_seq:.2f}초 / 동시에 {t_par:.2f}초")
```
</details>

In [ ]:
check("결과는 같다 (순서도 유지)", lambda: list(par) == seq)
check("동시에가 훨씬 빠르다", lambda: t_par < t_seq / 3, f"{t_par:.2f} vs {t_seq:.2f}")

## 2-6. Django ORM의 N+1 문제 ★
Django 없이 원리만 확인해요. SQLite에 **실제로 나간 쿼리 수**를 세요.

```python
# Django라면
for o in Order.objects.all()[:10]:       # 쿼리 1번
    print(o.user.name)                   # 주문마다 1번씩 → 10번  (N+1)
Order.objects.select_related("user")     # JOIN 한 번으로 해결
```

In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE users(id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE orders(id INTEGER PRIMARY KEY, user_id INTEGER REFERENCES users(id));
""")
db.executemany("INSERT INTO users VALUES (?,?)", [(i, f"회원{i}") for i in range(1, 6)])
db.executemany("INSERT INTO orders VALUES (?,?)", [(1000 + i, i % 5 + 1) for i in range(10)])
db.commit()

sent = []
db.set_trace_callback(sent.append)   # 실행된 SQL을 전부 기록

def n_plus_one():
    rows = db.execute("SELECT id, user_id FROM orders ORDER BY id").fetchall()
    return [(oid, db.execute("SELECT name FROM users WHERE id = ?", (uid,)).fetchone()[0]) for oid, uid in rows]

sent.clear(); slow = n_plus_one(); print("N+1 방식 쿼리 수:", len(sent))

In [ ]:
def with_join():
    return db.execute(___).fetchall()     # 쿼리 한 번으로 (주문id, 회원이름) 을 주문id 순으로

sent.clear(); fast = with_join(); print("JOIN 방식 쿼리 수:", len(sent))

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
def with_join():
    return db.execute("""
        SELECT o.id, u.name FROM orders o JOIN users u ON u.id = o.user_id ORDER BY o.id
    """).fetchall()

sent.clear(); fast = with_join(); print("JOIN 방식 쿼리 수:", len(sent))
```
</details>

In [ ]:
check("결과가 같다", lambda: fast == slow)
check("쿼리 1번", lambda: len(sent) == 1)

In [ ]:
django_quiz = {
    "MTV에서 DB 테이블 정의":             ___,   # "Model" / "Template" / "View"
    "MTV에서 요청을 받아 처리하는 함수":    ___,
    "MTV에서 HTML 화면":                 ___,
    "모델 변경 후 첫 명령":               ___,   # "python manage.py ..." 의 ... 부분
    "그 다음 실제 DB에 반영하는 명령":     ___,
    "FK(주문→회원) 한 번에 가져오기":      ___,   # "select_related" / "prefetch_related"
    "역방향·다대다(회원→주문들) 가져오기":  ___,
    "FastAPI의 Pydantic 모델 역할 (DRF)": ___,
}

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
django_quiz = {
    "MTV에서 DB 테이블 정의":             "Model",
    "MTV에서 요청을 받아 처리하는 함수":    "View",
    "MTV에서 HTML 화면":                 "Template",
    "모델 변경 후 첫 명령":               "makemigrations",
    "그 다음 실제 DB에 반영하는 명령":     "migrate",
    "FK(주문→회원) 한 번에 가져오기":      "select_related",     # SQL JOIN
    "역방향·다대다(회원→주문들) 가져오기":  "prefetch_related",   # 쿼리 2번 + 파이썬에서 합침
    "FastAPI의 Pydantic 모델 역할 (DRF)": "Serializer",
}
```
</details>

In [ ]:
check("Django", lambda: list(django_quiz.values()) == ["Model", "View", "Template", "makemigrations", "migrate",
                                                     "select_related", "prefetch_related", "Serializer"])

## 2-7. Linux
### (1) 권한 rwx ↔ 숫자
r=4, w=2, x=1을 더해서 소유자/그룹/기타 세 자리. `chmod 644` = `rw-r--r--`

In [ ]:
def perm_to_octal(perm):          # "rwxr-xr--" → "754"
    ___

def octal_to_perm(octal):         # "644" → "rw-r--r--"
    ___

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
def perm_to_octal(perm):          # "rwxr-xr--" → "754"
    val = {"r": 4, "w": 2, "x": 1, "-": 0}
    return "".join(str(sum(val[c] for c in perm[i:i + 3])) for i in (0, 3, 6))

def octal_to_perm(octal):         # "644" → "rw-r--r--"
    return "".join(("r" if int(d) & 4 else "-") + ("w" if int(d) & 2 else "-") + ("x" if int(d) & 1 else "-")
                   for d in octal)
```
</details>

In [ ]:
check("rwxr-xr-- → 754", lambda: perm_to_octal("rwxr-xr--") == "754")
check("rw-r--r-- → 644", lambda: perm_to_octal("rw-r--r--") == "644")
check("600 → rw-------", lambda: octal_to_perm("600") == "rw-------")
check("755 → rwxr-xr-x", lambda: octal_to_perm("755") == "rwxr-xr-x")

### (2) 표준 출력/에러 리다이렉트와 파이프 (실제 셸)
`ls`로 있는 폴더와 없는 폴더를 같이 보면 stdout과 stderr가 둘 다 나와요.
**둘 다** `out.log`에 들어가게 `___`를 채워요. 그리고 `app.log`에서 `ERROR` 줄 수를 세는 명령을 써요.

In [ ]:
import subprocess, tempfile, os
work = tempfile.mkdtemp()
def sh(cmd):
    r = subprocess.run(cmd, shell=True, cwd=work, capture_output=True, text=True, executable="/bin/bash")
    return r.stdout.strip()

with open(os.path.join(work, "app.log"), "w") as f:
    for i in range(50):
        f.write(f"2026-10-31 12:{i:02d} {'ERROR' if i % 7 == 0 else 'INFO'} request {i}\n")
print(sh("head -3 app.log"))

In [ ]:
sh("ls . /no_such_dir ___")          # stdout, stderr 모두 out.log로
error_count = sh(___)                 # app.log 의 ERROR 줄 수 (grep + 파이프 또는 grep 옵션)
print(sh("cat out.log")); print("ERROR 줄:", error_count)

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
sh("ls . /no_such_dir > out.log 2>&1")
error_count = sh("grep ERROR app.log | wc -l")     # 또는 "grep -c ERROR app.log"
print(sh("cat out.log")); print("ERROR 줄:", error_count)
```
</details>

In [ ]:
log = open(os.path.join(work, "out.log")).read() if os.path.exists(os.path.join(work, "out.log")) else ""
check("stdout(app.log)이 파일에", lambda: "app.log" in log)
check("stderr(No such file)도 파일에", lambda: "no_such_dir" in log)
check("ERROR 8줄", lambda: int(error_count) == 8)

### (3) 프로세스와 시그널
`sleep 30`을 백그라운드로 띄우고 PID를 확인한 뒤 종료 신호를 보내요.
파이썬에서 시그널로 죽은 프로세스의 `returncode`는 **-시그널번호**예요.

In [ ]:
import signal
p1 = subprocess.Popen(["sleep", "30"]); print("PID:", p1.pid)
p1.send_signal(___)        # `kill PID` 가 보내는 기본 신호 (정상 종료 요청)
p1.wait()

p2 = subprocess.Popen(["sleep", "30"])
p2.send_signal(___)        # `kill -9 PID` (강제, 마지막 수단)
p2.wait()
print(p1.returncode, p2.returncode)

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
import signal
p1 = subprocess.Popen(["sleep", "30"]); print("PID:", p1.pid)
p1.send_signal(signal.SIGTERM)
p1.wait()

p2 = subprocess.Popen(["sleep", "30"])
p2.send_signal(signal.SIGKILL)
p2.wait()
print(p1.returncode, p2.returncode)
```
</details>

In [ ]:
check("SIGTERM = 15", lambda: p1.returncode == -15)
check("SIGKILL = 9", lambda: p2.returncode == -9)

In [ ]:
linux_quiz = {
    "stdin, stdout, stderr 번호 (튜플)":   ___,
    "로그 파일이 주로 있는 폴더":            ___,
    "systemd 서비스 재시작 명령 (서비스 이름 agent)": ___,
    "그 서비스 로그 보기":                 ___,
    "버전이 맞아야 하는 설치 순서 (리스트)": ___,   # "PyTorch", "드라이버", "CUDA" 정렬
}

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
linux_quiz = {
    "stdin, stdout, stderr 번호 (튜플)":   (0, 1, 2),
    "로그 파일이 주로 있는 폴더":            "/var/log",
    "systemd 서비스 재시작 명령 (서비스 이름 agent)": "systemctl restart agent",
    "그 서비스 로그 보기":                 "journalctl -u agent",
    "버전이 맞아야 하는 설치 순서 (리스트)": ["드라이버", "CUDA", "PyTorch"],
}
```
</details>

In [ ]:
lq = list(linux_quiz.values())
check("표준 입출력 번호", lambda: lq[0] == (0, 1, 2))
check("/var/log", lambda: lq[1].rstrip("/") == "/var/log")
check("systemctl", lambda: lq[2].replace("sudo ", "") == "systemctl restart agent")
check("journalctl", lambda: lq[3].replace("sudo ", "") == "journalctl -u agent")
check("드라이버 → CUDA → PyTorch", lambda: lq[4] == ["드라이버", "CUDA", "PyTorch"])

## 2-8. Docker ★
### (1) 문제 있는 Dockerfile 고치기
아래 Dockerfile에는 문제가 **3개** 있어요. `lint()`가 아무 문제도 안 찾도록 `fixed`를 고쳐 써요.
1. 코드를 먼저 복사해서, 코드 한 줄만 바꿔도 `pip install` 층을 다시 만듦 (캐시 낭비)
2. 비밀키를 이미지에 박아 넣음
3. `127.0.0.1`로 열어서 컨테이너 밖에서 접속이 안 됨

In [ ]:
bad = """
FROM python:3.12-slim
WORKDIR /app
COPY . .
RUN pip install --no-cache-dir -r requirements.txt
ENV OPENAI_API_KEY=sk-live-123456
CMD ["uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"]
"""

def lint(dockerfile):
    lines = [l.strip() for l in dockerfile.strip().splitlines() if l.strip() and not l.strip().startswith("#")]
    find = lambda pred: next((i for i, l in enumerate(lines) if pred(l)), None)
    problems = []
    i_req = find(lambda l: l.startswith("COPY requirements.txt"))
    i_pip = find(lambda l: l.startswith("RUN pip install"))
    i_all = find(lambda l: l.replace(" ", "") in ("COPY..", "COPY./app"))
    if not lines or not lines[0].startswith("FROM"):
        problems.append("첫 줄은 FROM")
    if None in (i_req, i_pip, i_all) or not (i_req < i_pip < i_all):
        problems.append("COPY requirements.txt → RUN pip install → COPY . . 순서여야 캐시가 재사용돼요")
    if any(k in dockerfile.upper() for k in ("KEY=", "PASSWORD=", "SECRET=", "TOKEN=")):
        problems.append("비밀값은 Dockerfile 말고 --env-file 로")
    cmd = lines[-1] if lines else ""
    if not cmd.startswith("CMD") or "0.0.0.0" not in cmd:
        problems.append("마지막 CMD에서 --host 0.0.0.0 으로 열어야 밖에서 접속돼요")
    return problems

lint(bad)

In [ ]:
fixed = """
___
"""
lint(fixed)

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
fixed = """
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""
lint(fixed)
```
</details>

In [ ]:
check("Dockerfile 문제 0개", lambda: lint(fixed) == [], lint(fixed) if isinstance(fixed, str) else "")

### (2) `docker run` 명령 쓰기
이미지 `shop-agent`를 **백그라운드**로, 이름은 `agent`, 내 컴퓨터 **8080** → 컨테이너 **8000**, 환경변수는 `.env` 파일에서.

In [ ]:
run_cmd = ___

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
run_cmd = "docker run -d -p 8080:8000 --env-file .env --name agent shop-agent"
```
</details>

In [ ]:
parts = run_cmd.split() if isinstance(run_cmd, str) else []
check("docker run 으로 시작", lambda: parts[:2] == ["docker", "run"])
check("-d (백그라운드)", lambda: "-d" in parts)
check("-p 8080:8000 (호스트:컨테이너)", lambda: parts[parts.index("-p") + 1] == "8080:8000")
check("--env-file .env", lambda: parts[parts.index("--env-file") + 1] == ".env")
check("--name agent", lambda: parts[parts.index("--name") + 1] == "agent")
check("마지막은 이미지 이름", lambda: parts[-1] == "shop-agent")

### (3) docker compose 구성 (dict로)
`api`(현재 폴더 빌드, 8000 포트, `.env`, db에 의존)와 `db`(`mysql:8`, **DB 파일을 볼륨에 저장**)를 띄우는 compose 파일을 dict로 써요.

In [ ]:
compose = {
    "services": {
        "api": {"build": ___, "ports": ___, "env_file": ___, "depends_on": ___},
        "db":  {"image": ___, "environment": {"MYSQL_ROOT_PASSWORD": "dev", "MYSQL_DATABASE": "shop"},
                "volumes": ___},
    },
    "volumes": {"dbdata": {}},
}

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
compose = {
    "services": {
        "api": {"build": ".", "ports": ["8000:8000"], "env_file": ".env", "depends_on": ["db"]},
        "db":  {"image": "mysql:8", "environment": {"MYSQL_ROOT_PASSWORD": "dev", "MYSQL_DATABASE": "shop"},
                "volumes": ["dbdata:/var/lib/mysql"]},
    },
    "volumes": {"dbdata": {}},
}
```
</details>

In [ ]:
api, dbs = compose["services"]["api"], compose["services"]["db"]
check("api 빌드·포트", lambda: api["build"] == "." and api["ports"] == ["8000:8000"])
check("api env_file·depends_on", lambda: api["env_file"] in (".env", [".env"]) and api["depends_on"] == ["db"])
check("db 이미지", lambda: dbs["image"] == "mysql:8")
check("db 볼륨 (컨테이너를 지워도 데이터 유지)", lambda: dbs["volumes"] == ["dbdata:/var/lib/mysql"])

In [ ]:
docker_quiz = {
    "이미지를 만드는 레시피":                        ___,   # "Dockerfile" / "이미지" / "컨테이너" / "볼륨" / "레지스트리"
    "레시피로 만든 읽기 전용 틀":                     ___,
    "틀을 실행한 것 (지우면 안의 변경 내용도 사라짐)":   ___,
    "컨테이너가 지워져도 남아야 할 데이터를 바깥에 저장": ___,
    "Docker Hub, AWS ECR":                         ___,
    "VM보다 가벼운 이유: 호스트 OS의 ___를 공유":       ___,
}

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
docker_quiz = {
    "이미지를 만드는 레시피":                        "Dockerfile",
    "레시피로 만든 읽기 전용 틀":                     "이미지",
    "틀을 실행한 것 (지우면 안의 변경 내용도 사라짐)":   "컨테이너",
    "컨테이너가 지워져도 남아야 할 데이터를 바깥에 저장": "볼륨",
    "Docker Hub, AWS ECR":                         "레지스트리",
    "VM보다 가벼운 이유: 호스트 OS의 ___를 공유":       "커널",
}
```
</details>

In [ ]:
check("Docker 용어", lambda: list(docker_quiz.values()) == ["Dockerfile", "이미지", "컨테이너", "볼륨", "레지스트리", "커널"])

## 2-9. AWS · 클라우드

In [ ]:
aws = {
    "SSH로 접속하는 가상 서버 (GPU도 있음)":          ___,   # "EC2" "S3" "RDS" "IAM" "Lambda" "ECR" "CloudWatch" "VPC"
    "이미지·백업·모델 파일 저장 (버킷/키)":            ___,
    "백업·패치를 대신 해주는 관리형 MySQL/PostgreSQL": ___,
    "누가 무엇을 할 수 있는지 권한 관리":              ___,
    "서버 없이 함수만 올려 요청 때 실행":              ___,
    "Docker 이미지 저장소":                         ___,
    "로그·지표 모니터링과 알림":                      ___,
    "나만의 사설 네트워크":                          ___,
}
cloud_model = {
    "EC2, 네피리티 GPU 서버 (서버만 빌림)":  ___,   # "IaaS" / "PaaS" / "SaaS"
    "Render, Vercel (코드만 올리면 실행)":   ___,
    "완성된 고객지원 솔루션 구독":            ___,
}
# 웹 서버(HTTPS)와 MySQL이 있을 때, 보안그룹에서 '인터넷 전체(0.0.0.0/0)'에 열어도 되는 포트만
public_ports = {___}    # 후보: 22, 80, 443, 3306

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
aws = {
    "SSH로 접속하는 가상 서버 (GPU도 있음)":          "EC2",
    "이미지·백업·모델 파일 저장 (버킷/키)":            "S3",
    "백업·패치를 대신 해주는 관리형 MySQL/PostgreSQL": "RDS",
    "누가 무엇을 할 수 있는지 권한 관리":              "IAM",
    "서버 없이 함수만 올려 요청 때 실행":              "Lambda",
    "Docker 이미지 저장소":                         "ECR",
    "로그·지표 모니터링과 알림":                      "CloudWatch",
    "나만의 사설 네트워크":                          "VPC",
}
cloud_model = {
    "EC2, 네피리티 GPU 서버 (서버만 빌림)":  "IaaS",
    "Render, Vercel (코드만 올리면 실행)":   "PaaS",
    "완성된 고객지원 솔루션 구독":            "SaaS",
}
public_ports = {80, 443}   # 22(SSH)는 내 IP만, 3306(DB)은 절대 외부에 열지 않기
```
</details>

In [ ]:
check("AWS 서비스", lambda: list(aws.values()) == ["EC2", "S3", "RDS", "IAM", "Lambda", "ECR", "CloudWatch", "VPC"])
check("IaaS/PaaS/SaaS", lambda: list(cloud_model.values()) == ["IaaS", "PaaS", "SaaS"])
check("공개 포트는 80, 443만", lambda: public_ports == {80, 443})

## ✍️ 말로 설명해 보기 (30초)
1. 401과 403의 차이, 세션과 JWT의 차이는?
2. Django ORM의 N+1 문제와 해결법은?
3. Docker 이미지와 컨테이너의 차이, 볼륨이 필요한 이유는?
4. EC2·S3·RDS·IAM이 각각 무엇인가? IaaS/PaaS/SaaS 예시는?

<details><summary>모범 답안</summary>

1. 401은 누구인지 모를 때(인증 실패, 로그인 필요), 403은 누구인지는 알지만 권한이 없을 때(인가 실패)예요. 세션은 서버가 로그인 상태를 저장하고 브라우저엔 세션 ID만 두는 방식이라 강제 로그아웃이 쉽고, JWT는 서명된 토큰을 클라이언트가 들고 다녀서 서버 저장이 필요 없지만 만료 전에 무효화하기 어려워요.
2. 주문 N개를 가져온 뒤 주문마다 회원을 따로 조회해서 쿼리가 1+N번 나가는 문제예요. FK는 `select_related`(JOIN 한 번), 역방향·다대다는 `prefetch_related`(쿼리 2번)로 줄여요.
3. 이미지는 Dockerfile로 만든 읽기 전용 틀이고, 컨테이너는 그 이미지를 실행한 프로세스예요. 컨테이너를 지우면 안의 변경이 사라지니 DB 파일이나 모델 가중치는 볼륨으로 바깥에 저장해요.
4. EC2 가상 서버, S3 파일 저장소, RDS 관리형 DB, IAM 권한 관리. IaaS=EC2·GPU 서버, PaaS=Render·Vercel, SaaS=완성된 고객지원 솔루션.
</details>